# Pet Detection: YOLOv8 training on my own labels

Trains a YOLOv8n model on the 150 images I labeled in Label Studio, to check that the labels are usable for training.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

## 1. Install YOLOv8 (Ultralytics)

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 2. Get the labels from my GitHub repo

In [ ]:
!git clone -q https://github.com/sadia-mehrin-rahi/pet-detection-annotation.git
!ls pet-detection-annotation/labels/yolo/labels | wc -l

## 3. Upload the images
Zip your `my_images` folder (right-click → Send to → Compressed (zipped) folder), then upload `my_images.zip` when the button appears.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
!unzip -q -o "$zip_name" -d my_images_raw
!find my_images_raw -name '*.jpg' | wc -l

## 4. Match images to labels and split 80% train / 20% val
Label Studio adds an 8-character code to each label file name (e.g. `2b1453a2-beagle_67.txt`), so it is removed here to match `beagle_67.jpg`.

In [ ]:
import glob, os, re, random, shutil

label_dir = 'pet-detection-annotation/labels/yolo/labels'
images = {os.path.basename(p): p for p in glob.glob('my_images_raw/**/*.jpg', recursive=True)}

pairs = []
for lf in glob.glob(label_dir + '/*.txt'):
    base = re.sub(r'^[0-9a-f]{8}-', '', os.path.basename(lf))[:-4]
    if base + '.jpg' in images:
        pairs.append((images[base + '.jpg'], lf, base))
    else:
        print('No image for', base)
print('Matched pairs:', len(pairs))

random.seed(42)
random.shuffle(pairs)
n_val = int(0.2 * len(pairs))

shutil.rmtree('dataset', ignore_errors=True)
for i, (img, lbl, base) in enumerate(pairs):
    split = 'val' if i < n_val else 'train'
    os.makedirs(f'dataset/images/{split}', exist_ok=True)
    os.makedirs(f'dataset/labels/{split}', exist_ok=True)
    shutil.copy(img, f'dataset/images/{split}/{base}.jpg')
    shutil.copy(lbl, f'dataset/labels/{split}/{base}.txt')

print('Train:', len(os.listdir('dataset/images/train')), '| Val:', len(os.listdir('dataset/images/val')))

with open('data.yaml', 'w') as f:
    f.write('path: /content/dataset\ntrain: images/train\nval: images/val\nnames:\n  0: cat\n  1: dog\n')
print(open('data.yaml').read())

## 5. Train YOLOv8n (about 5–10 minutes on a T4 GPU)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')   # small model, pretrained on COCO
model.train(data='data.yaml', epochs=50, imgsz=640, batch=16, seed=42, name='pet')
save_dir = str(model.trainer.save_dir)
print('Results saved to:', save_dir)

## 6. Evaluate: mAP scores

In [ ]:
best = YOLO(save_dir + '/weights/best.pt')
m = best.val(data='data.yaml', split='val')

print('\n========== RESULTS ==========')
print(f'mAP@50      : {m.box.map50:.3f}')
print(f'mAP@50-95   : {m.box.map:.3f}')
print(f'Precision   : {m.box.mp:.3f}')
print(f'Recall      : {m.box.mr:.3f}')
for i, name in best.names.items():
    print(f'  {name:<4} mAP@50-95: {m.box.maps[i]:.3f}')

## 7. Look at the results

In [ ]:
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    p = os.path.join(save_dir, f)
    if os.path.exists(p):
        print(f); display(Image(filename=p, width=700))

## 8. Download results (for the GitHub repo)

In [ ]:
!mkdir -p training_results
!cp {save_dir}/results.png {save_dir}/confusion_matrix.png {save_dir}/val_batch0_pred.jpg training_results/ 2>/dev/null
!cp {save_dir}/results.csv training_results/
!zip -q -r training_results.zip training_results
files.download('training_results.zip')